# Clasificación de textos en Esperanto

**Minería de Textos — CUNEF 2026/2027**

Sistema completo para clasificar párrafos en Esperanto por temática (**ferrocarril, automóvil, aviación**):

1. **Fase 1 — Dataset**: obtención de párrafos etiquetados de Wikipedia en Esperanto.
2. **Fase 2 — NLP**: modelo propio de NLP basado en reglas, con API similar a spaCy.
3. **Fase 3 — Clasificación**: vectorización, entrenamiento, evaluación y clasificación de páginas web.

## 0. Configuración

Parámetros de ejecución del cuaderno:

* `GENERAR_DATASET = False` (por defecto): se usa el dataset **ya generado** y publicado en
  GitHub (ejecución asíncrona, sección 4.1.4 del enunciado). Es rápido y reproducible.
* `GENERAR_DATASET = True`: se vuelve a descargar todo de Wikipedia y se reconstruye el dataset
  (tarda ~1 minuto y depende de la conexión).

In [ ]:
GENERAR_DATASET = False     # True -> regenerar el dataset desde Wikipedia

DATASET_URL = ("https://raw.githubusercontent.com/guilletaff/Clasificacion_Textos_Esperanto/"
               "main/data/dataset_esperanto.csv")
RUTAS_LOCALES = ["../data/dataset_esperanto.csv", "data/dataset_esperanto.csv", "dataset_esperanto.csv"]
SALIDA_DATASET = "dataset_esperanto.csv"   # dónde se guarda el CSV si se regenera

import os
import re
import sys
import time
import subprocess

import pandas as pd
import matplotlib.pyplot as plt

try:                       # display() existe en Jupyter/Colab; fuera de ellos usamos print
    display
except NameError:
    display = print

# Fase 1 — Dataset

## 1.1 Temáticas elegidas

Tres categorías **semánticamente cercanas** dentro del dominio *transporte*, para que la
clasificación no sea trivial (comparten vocabulario: `motoro`, `rapido`, `pasaĝero`, `veturi`...):

| Clase | Significado | Artículos de Wikipedia (eo) |
|---|---|---|
| `fervojo` | ferrocarril | Fervojo, Lokomotivo, Metroo, Tramo, Rapidtrajno... |
| `automobilo` | automóvil | Aŭtomobilo, Kamiono, Aŭtobuso, Motociklo, Taksio... |
| `aviado` | aviación | Aviadilo, Aviado, Flughaveno, Helikoptero, Jetmotoro... |

## 1.2 Obtención del texto
Se usa la función `WikiPage` del repositorio guía ([`esperantilo`](https://github.com/jparisu/nlp-esperantilo)),
que consulta el API de Wikipedia. Si la librería no se puede instalar, se usa como **respaldo**
una llamada directa al API de MediaWiki con `requests`, para que el cuaderno sobreviva al fallo.

> **Importante:** si un título no existe en la Wikipedia en esperanto, `WikiPage.look_up` busca por
> defecto en otros idiomas. Así se colaron 22 párrafos **en español** (artículos *Komerca aviado* y
> *Supersona aviadilo*) en la primera versión del dataset. Por eso se usa `any_language=False` y se
> descarta cualquier página que no esté en esperanto.

In [ ]:
# CATEGORIAS: diccionario {clase: [títulos de artículos de Wikipedia en Esperanto]}
CATEGORIAS = {
    "fervojo": [
        "Fervojo",            # 'Ferrocarril'
        "Lokomotivo",         # 'Locomotora'
        "Metroo",             # 'Metro'
        "Tramo",              # 'Tranvía'
        "Rapidtrajno",        # 'Tren de alta velocidad'
        "Vagono",             # 'Vagón'
        "Fervoja stacio",     # 'Estación de ferrocarril'
        "Elektra lokomotivo", # 'Locomotora eléctrica'
        "Dizel-lokomotivo",   # 'Locomotora diésel'
        "Maglevo",            # 'Tren de levitación magnética'
    ],
    "automobilo": [
        "Aŭtomobilo",         # 'Automóvil'
        "Kamiono",            # 'Camión'
        "Aŭtobuso",           # 'Autobús'
        "Elektra aŭtomobilo", # 'Automóvil eléctrico'
        "Motociklo",          # 'Motocicleta'
        "Eksplodmotoro",      # 'Motor de combustión interna'
        "Taksio",             # 'Taxi'
        "Aŭtovojo",           # 'Autopista' / 'autovía'
        "Sportaŭto",          # 'Coche deportivo'
        "Karoserio",          # 'Carrocería'
    ],
    "aviado": [
        "Aviadilo",           # 'Avión'
        "Aviado",             # 'Aviación'
        "Flughaveno",         # 'Aeropuerto'
        "Helikoptero",        # 'Helicóptero'
        "Jetmotoro",          # 'Motor a reacción'
        "Komerca aviado",     # 'Aviación comercial'
        "Boeing 747",
        "Supersona aviadilo", # 'Avión supersónico'
        "Flugkompanio",       # 'Aerolínea'
        "Aeronaŭtiko",        # 'Aeronáutica'
    ],
}


PAUSA_ENTRE_PETICIONES = 1.0   # segundos; evita saturar el API de Wikipedia (error 429)
REINTENTOS = 4
WikiPage = None   # se rellena con preparar_fuente() solo si GENERAR_DATASET = True


def obtener_texto_wikipedia(titulo):
    """Devuelve el texto completo de un artículo de Wikipedia en Esperanto.

    1º intenta usar esperantilo.WikiPage (librería del repositorio guía);
    2º si no está disponible, llama directamente al API de MediaWiki.
    """
    if WikiPage is not None:
        # any_language=False: si el título no existe en esperanto NO se acepta la versión de
        # otro idioma (por defecto WikiPage la buscaría y colaría texto en español en el dataset)
        page = WikiPage.look_up(titulo, language="eo", any_language=False)
        if getattr(page, "language", "eo") != "eo":
            raise ValueError(f"'{titulo}' solo existe en '{page.language}', no en esperanto")
        return page.full_text()
    import requests
    for intento in range(REINTENTOS):
        r = requests.get(
            "https://eo.wikipedia.org/w/api.php",
            params={"action": "query", "prop": "extracts", "explaintext": 1, "redirects": 1,
                    "titles": titulo, "format": "json", "formatversion": 2},
            headers={"User-Agent": "esperanto-text-classification-student-project"},
            timeout=20,
        )
        if r.status_code != 429:          # 429 = demasiadas peticiones -> esperar y reintentar
            break
        espera = int(r.headers.get("Retry-After", 2 ** (intento + 1)))
        print(f"    [429] Wikipedia pide esperar {espera} s (intento {intento + 1}/{REINTENTOS})")
        time.sleep(espera)
    r.raise_for_status()
    pagina = r.json()["query"]["pages"][0]
    if "missing" in pagina:
        raise ValueError(f"El artículo '{titulo}' no existe")
    return pagina["extract"]


def preparar_fuente():
    """Importa esperantilo; si no está instalado lo instala desde GitHub. Devuelve WikiPage o None."""
    try:
        from esperantilo import WikiPage as wp
        return wp
    except ImportError:
        pass
    try:
        print("Instalando esperantilo desde GitHub...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "git+https://github.com/jparisu/nlp-esperantilo.git"], check=True, timeout=300)
        from esperantilo import WikiPage as wp
        return wp
    except Exception as e:  # noqa: BLE001
        print(f"[aviso] esperantilo no disponible ({type(e).__name__}); se usará el API de MediaWiki directamente.")
        return None

## 1.3 Limpieza y segmentación en párrafos

Cada artículo se divide por saltos de línea (un párrafo por línea) y cada párrafo se limpia
conforme al **contrato de datos**: sin cabeceras de sección (`== Historio ==`), sin el carácter `;`
(rompería el CSV) y sin espacios redundantes. Se descartan fragmentos de menos de 40 caracteres
(títulos sueltos, elementos de listas vacíos...).

Cada párrafo **hereda la etiqueta del artículo**, lo que introduce ruido: hay párrafos genéricos
que no distinguen la categoría. Ese ruido se mide más adelante con el modelo NLP (sección 2.9).

In [ ]:
LONGITUD_MINIMA = 40


def limpiar_parrafo(parrafo: str) -> str:
    """Limpia un párrafo conforme al contrato de datos."""
    texto = re.sub(r"==+.*?==+", "", parrafo)   # cabeceras de sección
    texto = texto.replace(";", ",")             # el ';' es el separador del CSV
    texto = re.sub(r"\s+", " ", texto).strip()  # espacios y saltos de línea
    return texto


def construir_dataset():
    """Descarga los artículos de CATEGORIAS, los divide en párrafos y devuelve un DataFrame text;class.

    Si un artículo falla (red, título inexistente...) se avisa y se continúa con el resto.
    """
    filas = []
    print("Iniciando la recopilación de artículos de Wikipedia en Esperanto...\n")
    for categoria, articulos in CATEGORIAS.items():
        print(f"--- Procesando categoría: {categoria} ---")
        parrafos_clase = 0
        for titulo in articulos:
            try:
                texto_completo = obtener_texto_wikipedia(titulo)
                for p in texto_completo.split("\n"):
                    p_limpio = limpiar_parrafo(p)
                    if len(p_limpio) >= LONGITUD_MINIMA:
                        filas.append({"text": p_limpio, "class": categoria})
                        parrafos_clase += 1
                print(f"  [OK] {titulo} - párrafos acumulados en la clase: {parrafos_clase}")
            except Exception as e:  # noqa: BLE001
                print(f"  [Error] No se pudo obtener '{titulo}': {e}")
            time.sleep(PAUSA_ENTRE_PETICIONES)
        print()
    return pd.DataFrame(filas, columns=["text", "class"])


def cargar_dataset():
    """Lee el dataset ya generado: primero de GitHub y, si falla, de una copia local."""
    for fuente in [DATASET_URL] + RUTAS_LOCALES:
        try:
            datos = pd.read_csv(fuente, sep=";", encoding="utf-8")
            print(f"Dataset cargado desde: {fuente}")
            return datos
        except Exception as e:  # noqa: BLE001
            print(f"[aviso] No se pudo leer {fuente}: {type(e).__name__}")
    raise RuntimeError("No se encontró el dataset en ninguna fuente")

## 1.4 Generación o carga del dataset
Según `GENERAR_DATASET`, se regenera desde Wikipedia o se carga la versión publicada.
Si la regeneración falla (p. ej. sin conexión, o alguna clase queda sin párrafos), se recurre a la
versión publicada, para que el resto del cuaderno pueda ejecutarse igualmente.

In [ ]:
df = None
if GENERAR_DATASET:
    WikiPage = preparar_fuente()
    df = construir_dataset()
    clases_vacias = set(CATEGORIAS) - set(df["class"])
    if clases_vacias:
        # Un dataset al que le falta una clase no sirve: mejor usar la versión publicada
        print(f"[aviso] Sin párrafos para {sorted(clases_vacias)}; se carga el dataset publicado.")
        df = None
    else:
        df.to_csv(SALIDA_DATASET, sep=";", index=False, encoding="utf-8")
        print(f"Dataset generado y guardado en: {SALIDA_DATASET}")
if df is None:
    df = cargar_dataset()

# Comprobación del contrato de datos
assert list(df.columns) == ["text", "class"], "Cabecera distinta de text;class"
assert not df["text"].str.contains(";").any(), "Hay párrafos con ';'"
print(f"\nTotal de párrafos: {len(df)}")
print(df["class"].value_counts())
df.head()

# Fase 2 — Modelo de NLP para Esperanto (`esperantilo-nlp`)

En esta fase construimos **desde cero** un modelo de NLP para Esperanto, **basado en reglas**
y con un **API inspirado en spaCy**. No se usa ninguna librería de NLP existente
(ni spaCy, ni NLTK, ni Stanza): solo la librería estándar de Python (`re`, `unicodedata`, ...).

### ¿Por qué reglas y no aprendizaje automático?
El Esperanto es una lengua **planificada** con una morfología **regular y aglutinante**:
la categoría gramatical de casi cualquier palabra se deduce de su **terminación**.

| Terminación | Categoría | Ejemplo | Traducción |
|---|---|---|---|
| `-o` | sustantivo | `trajno` | tren |
| `-a` | adjetivo | `rapida` | rápido |
| `-e` | adverbio derivado | `rapide` | rápidamente |
| `-i` | infinitivo | `veturi` | circular / viajar |
| `-as` / `-is` / `-os` | presente / pasado / futuro | `veturas`, `veturis`, `veturos` | |
| `-us` / `-u` | condicional / volitivo | `veturus`, `veturu` | |
| `-j` | plural (sust./adj.) | `trajnoj` | trenes |
| `-n` | acusativo (objeto directo / dirección) | `trajnon` | |
| `-ant-`, `-int-`, `-ont-` | participios activos (pres./pas./fut.) | `veturanta` | que circula |
| `-at-`, `-it-`, `-ot-` | participios pasivos (pres./pas./fut.) | `konstruita` | construido |

Lo único que **no** sigue estas reglas es un conjunto **cerrado y pequeño** de palabras
(artículo, pronombres, correlativos, preposiciones, conjunciones, numerales, partículas...),
que se recogen en un **léxico** escrito a mano.

### Arquitectura (pipeline al estilo spaCy)

```
texto ──► Tokenizer ──► normalizer ──► sentencizer ──► tagger ──► lemmatizer ──► stopwords ──► Doc
          (tokens)     (norm_, x-sistemo) (is_sent_start) (pos_, tag_, morph) (lemma_, _.radiko) (is_stop)
```

| spaCy | Este modelo |
|---|---|
| `nlp = spacy.load("...")` | `nlp = load("eo")` o `nlp = Esperanto()` |
| `doc = nlp(texto)` | `doc = nlp(texto)` |
| `token.text`, `token.lemma_`, `token.pos_`, `token.tag_`, `token.morph` | idem |
| `token.is_stop`, `token.is_punct`, `token.like_num`, `token.shape_` ... | idem |
| `doc.sents`, `doc.noun_chunks`, `doc[2:5]` (`Span`) | idem |
| `nlp.pipe(textos)`, `nlp.pipe_names`, `nlp.add_pipe`, `nlp.select_pipes` | idem |
| `nlp.Defaults.stop_words` | idem |
| `Token.set_extension(...)` / `token._.x` | idem |
| `spacy.explain("NOUN")` | `explain("NOUN")` |

Las etiquetas POS siguen el estándar **Universal Dependencies (UPOS)** y los rasgos
morfológicos el formato UD (`Case=Acc|Number=Plur`), igual que spaCy.

In [ ]:
# ============================================================================
# Imports: solo librería estándar (+ requests para descargar las stop-words)
# ============================================================================
import re
import json
import time
import unicodedata
from collections import Counter
from contextlib import contextmanager
from functools import lru_cache

try:
    import requests  # solo se usa para descargar (opcionalmente) vortoj.json
except ImportError:  # el modelo funciona igual sin conexión
    requests = None

## 2.1 Recursos lingüísticos: léxico de palabras cerradas

Estas son las palabras que **no** se pueden analizar por su terminación. Para cada una se
guarda su categoría UPOS, su lema y sus rasgos morfológicos.

* **Artículo**: solo existe `la` (y su forma elidida `l'`). No hay artículo indefinido.
* **Pronombres personales**: `mi, vi, li, ŝi, ĝi, ni, ili, oni, si, ci`. A partir de cada uno
  se generan **automáticamente** el acusativo (`+n` → `lin`) y los posesivos
  (`+a/+aj/+an/+ajn` → `mia, miaj, mian, miajn`).
* **Correlativos**: la famosa tabla de 5 × 9 (`ki-/ti-/i-/ĉi-/neni-` × `-u/-o/-a/-e/-am/-el/-al/-om/-es`)
  también se **genera por reglas**, junto con sus formas flexionadas (`kiuj`, `tiun`, `kion`, `tien`...).
* **Preposiciones, conjunciones, numerales, partículas, adverbios primitivos e interjecciones**:
  listas cerradas.

In [ ]:
# ----------------------------------------------------------------------------
# Pronombres personales: forma base -> rasgos morfológicos
# ----------------------------------------------------------------------------
PRONOMOJ = {
    "mi":  {"Person": "1", "Number": "Sing"},
    "ci":  {"Person": "2", "Number": "Sing"},
    "vi":  {"Person": "2"},                      # singular o plural: ambiguo
    "li":  {"Person": "3", "Number": "Sing", "Gender": "Masc"},
    "ŝi":  {"Person": "3", "Number": "Sing", "Gender": "Fem"},
    "ĝi":  {"Person": "3", "Number": "Sing", "Gender": "Neut"},
    "ni":  {"Person": "1", "Number": "Plur"},
    "ili": {"Person": "3", "Number": "Plur"},
    "oni": {"Person": "3", "PronType": "Ind"},   # impersonal ("se")
    "si":  {"Person": "3", "Reflex": "Yes"},     # reflexivo
}

# ----------------------------------------------------------------------------
# Tabla de correlativos: 5 comienzos x 9 finales
# ----------------------------------------------------------------------------
KOREL_KOMENCOJ = {          # comienzo -> PronType (UD)
    "ki": "Int,Rel",        # interrogativo / relativo   (kiu = quién/cuál)
    "ti": "Dem",            # demostrativo               (tiu = ese)
    "i": "Ind",             # indefinido                 (iu = alguien)
    "ĉi": "Tot",            # colectivo / universal      (ĉiu = cada/todos)
    "neni": "Neg",          # negativo                   (neniu = nadie)
}
KOREL_FINOJ = {             # final -> (UPOS por defecto, significado)
    "u": ("PRON", "individuo"),   # kiu, tiu, ĉiu...
    "o": ("PRON", "aĵo"),         # kio, tio, ĉio...
    "a": ("DET", "eco"),          # kia, tia, ĉia...
    "es": ("DET", "posedo"),      # kies, ties, ĉies...
    "e": ("ADV", "loko"),         # kie, tie, ĉie...
    "am": ("ADV", "tempo"),       # kiam, tiam, ĉiam...
    "el": ("ADV", "maniero"),     # kiel, tiel, ĉiel...
    "al": ("ADV", "kaŭzo"),       # kial, tial, ĉial...
    "om": ("ADV", "kvanto"),      # kiom, tiom, ĉiom...
}

# ----------------------------------------------------------------------------
# Resto de palabras cerradas
# ----------------------------------------------------------------------------
PREPOZICIOJ = {
    "al", "anstataŭ", "antaŭ", "apud", "ĉe", "ĉirkaŭ", "da", "de", "dum", "ekde", "ekster",
    "el", "en", "ĝis", "inter", "je", "kontraŭ", "krom", "kun", "laŭ", "malantaŭ",
    "malgraŭ", "per", "po", "por", "post", "preter", "pri", "pro", "sen", "sub", "super",
    "sur", "tra", "trans", "ĉirkaŭ", "kiel",  # "kiel" se re-etiqueta por contexto (ver tagger)
} - {"kiel"}
KONJUNKCIOJ_KUNORDIGAJ = {"kaj", "aŭ", "sed", "nek"}                    # CCONJ
KONJUNKCIOJ_SUBORDIGAJ = {"ke", "se", "ĉar", "kvankam", "kvazaŭ", "ol"}  # SCONJ
PARTIKULOJ = {"ne", "ja", "jen", "ajn", "ĉi", "ĉu", "ju", "des", "mem"}  # PART
ADVERBOJ_PRIMITIVAJ = {     # adverbios que NO terminan en -e
    "ankaŭ", "ankoraŭ", "almenaŭ", "apenaŭ", "baldaŭ", "hieraŭ", "hodiaŭ", "morgaŭ",
    "preskaŭ", "nun", "jam", "tre", "tro", "tuj", "for", "plu", "pli", "plej", "nur",
    "eĉ", "ĵus", "do", "tamen", "ambaŭ", "jes",
}
INTERJEKCIOJ = {"adiaŭ", "ha", "he", "ho", "hu", "nu", "ve", "fi", "hura", "bis", "ek"}
NOMBROJ_BAZAJ = {"nul": 0, "unu": 1, "du": 2, "tri": 3, "kvar": 4, "kvin": 5, "ses": 6,
                 "sep": 7, "ok": 8, "naŭ": 9, "dek": 10, "cent": 100, "mil": 1000}
# Numerales compuestos escritos juntos: dudek (20), tricent (300), kvarmil (4000)...
NOMBRO_RE = re.compile(r"^(?:(?:du|tri|kvar|kvin|ses|sep|ok|naŭ)?(?:dek|cent|mil))+$")

# Verbos que se comportan como auxiliares (AUX en UD), igual que "be/can/must/want" en inglés
HELPVERBOJ = {"esti", "povi", "devi", "voli"}

# Abreviaturas frecuentes en Wikipedia: forma -> (norma expandida, UPOS)
MALLONGIGOJ = {
    "ekz.": ("ekzemple", "ADV"), "k.t.p.": ("kaj tiel plu", "ADV"), "ktp.": ("kaj tiel plu", "ADV"),
    "ktp": ("kaj tiel plu", "ADV"), "k.s.": ("kaj simile", "ADV"), "ks.": ("kaj simile", "ADV"),
    "t.e.": ("tio estas", "ADV"), "t.n.": ("tiel nomata", "ADJ"), "ĉ.": ("ĉirkaŭ", "ADP"),
    "p.k.": ("post kristo", "ADV"), "a.k.": ("antaŭ kristo", "ADV"), "s-ro": ("sinjoro", "NOUN"),
    "s-ino": ("sinjorino", "NOUN"), "d-ro": ("doktoro", "NOUN"), "f-ino": ("fraŭlino", "NOUN"),
    "n-ro": ("numero", "NOUN"), "km": ("kilometro", "NOUN"), "kg": ("kilogramo", "NOUN"),
    "mm": ("milimetro", "NOUN"), "cm": ("centimetro", "NOUN"), "m": ("metro", "NOUN"),
    "h": ("horo", "NOUN"), "t": ("tuno", "NOUN"), "kw": ("kilovato", "NOUN"),
    "kv": ("kilovolto", "NOUN"), "km/h": ("kilometroj hore", "NOUN"),
}
# Abreviaturas que pueden cerrar una frase ("... trajnoj, aŭtoj ktp. La ...")
MALLONGIGOJ_FINAJ = {"ktp.", "k.t.p.", "ks.", "k.s."}

### Construcción del léxico
A partir de las listas anteriores se genera un diccionario `forma -> análisis`.
Lo interesante es que **pronombres y correlativos se flexionan por reglas**, de modo que no
hay que escribir a mano `kiujn`, `tiun`, `iliajn`...

In [ ]:
def _konstrui_leksikonon():
    """Devuelve un dict {forma: (pos, lema, rasgos, tag)} con todas las palabras cerradas."""
    lex = {}

    def aldoni(forma, pos, lemo, morph, tag):
        lex.setdefault(forma, (pos, lemo, dict(morph), tag))

    # Artículo definido (y su forma elidida l')
    aldoni("la", "DET", "la", {"Definite": "Def", "PronType": "Art"}, "artikolo")
    aldoni("l'", "DET", "la", {"Definite": "Def", "PronType": "Art"}, "artikolo")

    # Pronombres personales + acusativo + posesivos
    for baz, feats in PRONOMOJ.items():
        prontype = feats.get("PronType", "Prs")
        aldoni(baz, "PRON", baz, {**feats, "PronType": prontype, "Case": "Nom"}, "pronomo")
        aldoni(baz + "n", "PRON", baz, {**feats, "PronType": prontype, "Case": "Acc"}, "pronomo")
        for fino, extra in (("a", {"Number[psor]": None, "Case": "Nom"}),
                            ("aj", {"Number": "Plur", "Case": "Nom"}),
                            ("an", {"Case": "Acc"}),
                            ("ajn", {"Number": "Plur", "Case": "Acc"})):
            f = {k: v for k, v in feats.items() if k not in ("Number",)}
            f.update({"Poss": "Yes", "PronType": "Prs"})
            f.update({k: v for k, v in extra.items() if v is not None})
            aldoni(baz + fino, "DET", baz + "a", f, "pronomo-poseda")

    # Tabla de correlativos y sus formas flexionadas
    for kom, prontype in KOREL_KOMENCOJ.items():
        for fin, (pos, _sign) in KOREL_FINOJ.items():
            baz = kom + fin
            f = {"PronType": prontype}
            if fin == "es":
                f["Poss"] = "Yes"
            aldoni(baz, pos, baz, {**f, **({"Case": "Nom"} if fin in "uao" else {})}, "korelativo")
            if fin in ("u", "a"):          # kiu, kiuj, kiun, kiujn
                aldoni(baz + "j", pos, baz, {**f, "Number": "Plur", "Case": "Nom"}, "korelativo")
                aldoni(baz + "n", pos, baz, {**f, "Case": "Acc"}, "korelativo")
                aldoni(baz + "jn", pos, baz, {**f, "Number": "Plur", "Case": "Acc"}, "korelativo")
            elif fin == "o":               # kio, kion
                aldoni(baz + "n", pos, baz, {**f, "Case": "Acc"}, "korelativo")
            elif fin == "e":               # kie -> kien (dirección)
                aldoni(baz + "n", pos, baz, {**f, "Case": "Acc"}, "korelativo")

    for w in PREPOZICIOJ:
        aldoni(w, "ADP", w, {}, "prepozicio")
    for w in KONJUNKCIOJ_KUNORDIGAJ:
        aldoni(w, "CCONJ", w, {}, "konjunkcio")
    for w in KONJUNKCIOJ_SUBORDIGAJ:
        aldoni(w, "SCONJ", w, {}, "konjunkcio")
    for w in PARTIKULOJ:
        aldoni(w, "PART", w, {"Polarity": "Neg"} if w == "ne" else {}, "partikulo")
    for w in ADVERBOJ_PRIMITIVAJ:
        aldoni(w, "ADV", w, {}, "adverbo")
    for w in INTERJEKCIOJ:
        aldoni(w, "INTJ", w, {}, "interjekcio")
    for w in NOMBROJ_BAZAJ:
        aldoni(w, "NUM", w, {"NumType": "Card"}, "nombro")
    for w, (norm, pos) in MALLONGIGOJ.items():
        aldoni(w, pos, norm, {"Abbr": "Yes"}, "mallongigo")
    return lex


LEKSIKONO = _konstrui_leksikonon()
print(f"Léxico de palabras cerradas: {len(LEKSIKONO)} formas")

## 2.2 Stop-words

Se parte de la lista de referencia del repositorio guía
([`vortoj.json`](https://github.com/jparisu/nlp-esperantilo/blob/main/resources/esperanto/vortoj.json)),
que se **descarga** de GitHub. Para que el cuaderno sea **autocontenido** y sobreviva a un fallo
de red, se incluye además una **copia embebida** de la lista: si la descarga falla, se usa la copia.

Decisiones de diseño:
* Solo se usan las entradas con `"ignorinda": true` (se descartan `bona`, `feliĉa`, `miliono`,
  `miliardo`, que sí aportan significado).
* Se **añaden** todos los correlativos (`kiu`, `tio`, `ĉiuj`...), que faltan en la lista original
  y son tan vacíos de contenido temático como un pronombre.
* La comprobación se hace sobre la **forma normalizada _y_ sobre el lema**: así `estas`, `estis`,
  `havas`, `lin`, `ilia`... son stop-words aunque en la lista solo aparezcan `esti`, `havi`, `li`, `ili`.

In [ ]:
VORTOJ_URL = ("https://raw.githubusercontent.com/jparisu/nlp-esperantilo/"
              "main/resources/esperanto/vortoj.json")

# Copia embebida de las entradas "ignorinda": true de vortoj.json (122 palabras)
HALTVORTOJ_ENIGITAJ = set("""
adiaŭ ajn al ankaŭ ankoraŭ anstataŭ antaŭ apenaŭ apud aŭ baldaŭ bonvoli cent ci da de dek
des do doktoro du dum ekster ekzemple el en esti eĉ eble for fraŭlo ha havi he hieraŭ ho
hodiaŭ hu ili inter ja jam je jen jes ju kaj ke kontraŭ krom kun kvankam kvar kvazaŭ kvin la
laŭ li male malgraŭ mem mi mil minus morgaŭ naŭ ne nek nepre ni nu nul nun nur ok ol oni per
plej pli plu plus po por post povi preskaŭ preter pri pro se sed sen sep ses si sinjoro sub
super supre sur tamen tra trans tre tri tro tuj tute unu ve verŝajne vi ĉar ĉe ĉi ĉirkaŭ ĉu
ĝi ĝis ĵus ŝi
""".split())


def ŝargi_haltvortojn(url=VORTOJ_URL, timeout=10):
    """Descarga vortoj.json; si falla (sin red, 404, JSON corrupto...) usa la copia embebida."""
    vortoj = set(HALTVORTOJ_ENIGITAJ)
    fonto = "copia embebida"
    if requests is not None:
        try:
            r = requests.get(url, timeout=timeout)
            r.raise_for_status()
            datumoj = r.json()
            elŝutitaj = {e["vorto"].lower() for e in datumoj["vortoj"] if e.get("ignorinda", True)}
            if elŝutitaj:
                vortoj = elŝutitaj
                fonto = "GitHub (vortoj.json)"
        except Exception as e:  # noqa: BLE001 - cualquier fallo de red/formato
            print(f"[aviso] No se pudo descargar vortoj.json ({type(e).__name__}); uso la copia embebida.")
    # Añadimos los correlativos base (kiu, tio, ĉiuj se cubren vía lema)
    vortoj |= {k + f for k in KOREL_KOMENCOJ for f in KOREL_FINOJ}
    # Y el resto de palabras cerradas que puedan faltar en la lista
    vortoj |= PREPOZICIOJ | KONJUNKCIOJ_KUNORDIGAJ | KONJUNKCIOJ_SUBORDIGAJ | PARTIKULOJ
    vortoj |= set(PRONOMOJ) | {"l'"} | HELPVERBOJ
    return vortoj, fonto


HALTVORTOJ, _fonto = ŝargi_haltvortojn()
print(f"Stop-words cargadas: {len(HALTVORTOJ)} (fuente: {_fonto})")

## 2.3 Objetos `MorphAnalysis`, `Token`, `Span` y `Doc`

Estructuran el resultado igual que en spaCy:
* `Doc` es la secuencia de `Token`s de un texto. Se puede iterar, indexar (`doc[0]`) y
  trocear (`doc[1:4]` devuelve un `Span`).
* Los atributos de texto terminan en `_` (`lemma_`, `pos_`...) como en spaCy.
* `Token.set_extension()` permite añadir atributos propios accesibles con `token._.nombre`.

In [ ]:
class MorphAnalysis:
    """Rasgos morfológicos en formato Universal Dependencies (p. ej. 'Case=Acc|Number=Plur')."""

    def __init__(self, feats=None):
        self._feats = {k: v for k, v in (feats or {}).items() if v is not None}

    def get(self, field):
        """Como en spaCy: devuelve una lista con los valores del rasgo (vacía si no existe)."""
        return self._feats[field].split(",") if field in self._feats else []

    def to_dict(self):
        return dict(self._feats)

    def __contains__(self, item):
        return item in str(self).split("|")

    def __len__(self):
        return len(self._feats)

    def __str__(self):
        return "|".join(f"{k}={v}" for k, v in sorted(self._feats.items()))

    def __repr__(self):
        return str(self)

    def __eq__(self, other):
        return str(self) == str(other)


class _Underscore:
    """Contenedor de atributos personalizados (token._.xxx / doc._.xxx)."""

    def __init__(self, extensions, obj):
        object.__setattr__(self, "_ext", extensions)
        object.__setattr__(self, "_obj", obj)
        object.__setattr__(self, "_vals", {})

    def __getattr__(self, name):
        if name not in self._ext:
            raise AttributeError(f"Extensión '{name}' no registrada. Usa set_extension('{name}', ...)")
        default, getter = self._ext[name]
        if getter is not None:
            return getter(self._obj)
        return self._vals.get(name, default)

    def __setattr__(self, name, value):
        if name not in self._ext:
            raise AttributeError(f"Extensión '{name}' no registrada.")
        self._vals[name] = value


class _Extensible:
    """Añade el mecanismo set_extension()/._ de spaCy a una clase."""
    _extensions = None

    @classmethod
    def set_extension(cls, name, default=None, getter=None, force=False):
        if cls._extensions is None:
            cls._extensions = {}
        if name in cls._extensions and not force:
            raise ValueError(f"La extensión '{name}' ya existe (usa force=True).")
        cls._extensions[name] = (default, getter)

    @classmethod
    def has_extension(cls, name):
        return bool(cls._extensions) and name in cls._extensions

    @property
    def _(self):
        if "_underscore" not in self.__dict__:
            self.__dict__["_underscore"] = _Underscore(type(self)._extensions or {}, self)
        return self.__dict__["_underscore"]


class Token(_Extensible):
    """Una palabra, signo de puntuación o símbolo dentro de un Doc."""
    _extensions = {}

    def __init__(self, doc, i, text, idx, whitespace):
        self.doc = doc
        self.i = i                      # posición en el Doc
        self.text = text                # texto original
        self.idx = idx                  # offset (carácter) en el texto original
        self.whitespace_ = whitespace   # espacio que sigue al token
        self.norm_ = text.lower()       # forma normalizada (la rellena el normalizer)
        self.lemma_ = ""
        self.pos_ = ""
        self.tag_ = ""
        self.morph = MorphAnalysis()
        self.is_stop = False
        self.is_sent_start = None

    # --- Atributos derivados (calculados a partir del texto) ---------------
    @property
    def text_with_ws(self):
        return self.text + self.whitespace_

    @property
    def lower_(self):
        return self.text.lower()

    @property
    def is_alpha(self):
        return self.text.replace("'", "").replace("’", "").isalpha()

    @property
    def is_digit(self):
        return self.text.isdigit()

    @property
    def is_punct(self):
        return self.pos_ == "PUNCT"

    @property
    def is_space(self):
        return self.text.isspace()

    @property
    def is_title(self):
        return self.text.istitle()

    @property
    def is_upper(self):
        return self.text.isupper()

    @property
    def like_num(self):
        t = self.norm_.replace(".", "").replace(",", "")
        return t.isdigit() or self.pos_ == "NUM" or bool(NOMBRO_RE.match(self.norm_))

    @property
    def like_url(self):
        return bool(re.match(r"^(https?://|www\.)", self.text, re.I))

    @property
    def shape_(self):
        """Forma ortográfica: 'Trajno' -> 'Xxxxx' (máx. 4 repeticiones seguidas, como spaCy)."""
        s = "".join("X" if c.isupper() else "x" if c.isalpha() else "d" if c.isdigit() else c
                    for c in self.text)
        return re.sub(r"(.)\1{4,}", lambda m: m.group(1) * 4, s)

    @property
    def is_sent_end(self):
        return self.i + 1 == len(self.doc) or bool(self.doc[self.i + 1].is_sent_start)

    @property
    def sent(self):
        for s in self.doc.sents:
            if s.start <= self.i < s.end:
                return s

    def nbor(self, i=1):
        return self.doc[self.i + i]

    def __len__(self):
        return len(self.text)

    def __str__(self):
        return self.text

    def __repr__(self):
        return self.text


class Span:
    """Fragmento contiguo de un Doc: doc[inicio:fin]."""

    def __init__(self, doc, start, end, label=""):
        self.doc, self.start, self.end, self.label_ = doc, start, end, label

    def __iter__(self):
        return (self.doc[i] for i in range(self.start, self.end))

    def __len__(self):
        return self.end - self.start

    def __getitem__(self, i):
        toks = list(self)
        return toks[i]

    @property
    def text(self):
        return "".join(t.text_with_ws for t in self).strip()

    @property
    def lemma_(self):
        return " ".join(t.lemma_ for t in self)

    @property
    def root(self):
        """Sin análisis sintáctico: aproximamos el núcleo como el último sustantivo/nombre propio."""
        for t in reversed(list(self)):
            if t.pos_ in ("NOUN", "PROPN"):
                return t
        return self.doc[self.end - 1]

    def __str__(self):
        return self.text

    def __repr__(self):
        return self.text


class Doc(_Extensible):
    """Secuencia de Tokens producida por nlp(texto)."""
    _extensions = {}

    def __init__(self, text, tokens_info):
        self.text = text
        self._tokens = [Token(self, i, t, idx, ws) for i, (t, idx, ws) in enumerate(tokens_info)]

    def __iter__(self):
        return iter(self._tokens)

    def __len__(self):
        return len(self._tokens)

    def __getitem__(self, key):
        if isinstance(key, slice):
            start, stop, _ = key.indices(len(self))
            return Span(self, start, stop)
        return self._tokens[key]

    @property
    def sents(self):
        """Generador de frases (Span). Requiere el componente 'sentencizer'."""
        start = 0
        for t in self._tokens[1:]:
            if t.is_sent_start:
                yield Span(self, start, t.i)
                start = t.i
        if self._tokens:
            yield Span(self, start, len(self))

    @property
    def noun_chunks(self):
        """Sintagmas nominales simples: (DET|NUM)* (ADJ)* (NOUN|PROPN)+ dentro de una frase.

        Gracias a la concordancia del Esperanto (el adjetivo lleva las mismas -j/-n que el
        sustantivo), exigimos que los ADJ concuerden en número y caso con el núcleo.
        """
        toks, n, i = self._tokens, len(self._tokens), 0
        while i < n:
            j = i
            while j < n and toks[j].pos_ in ("DET", "NUM"):
                j += 1
            k = j
            while k < n and toks[k].pos_ == "ADJ":
                k += 1
            m = k
            while m < n and toks[m].pos_ in ("NOUN", "PROPN") and (m == k or not toks[m].is_sent_start):
                m += 1
            if m > k:
                nucleo = toks[k]
                kongruas = all(t.morph.get("Number") == nucleo.morph.get("Number") and
                               t.morph.get("Case") == nucleo.morph.get("Case") for t in toks[j:k])
                start = i if kongruas else k
                yield Span(self, start, m, label="NP")
                i = m
            else:
                i = max(i + 1, k)

    def count_by(self, attr):
        """Frecuencia de un atributo, p. ej. doc.count_by('pos_')."""
        return Counter(getattr(t, attr) for t in self._tokens)

    def to_dataframe(self):
        """Tabla con el análisis de cada token (útil para visualizar)."""
        import pandas as pd
        return pd.DataFrame([{
            "text": t.text, "norm": t.norm_, "lemma": t.lemma_, "pos": t.pos_, "tag": t.tag_,
            "morph": str(t.morph), "radiko": t._.radiko if Token.has_extension("radiko") else "",
            "is_stop": t.is_stop, "is_punct": t.is_punct, "sent_start": bool(t.is_sent_start),
        } for t in self._tokens])

    def to_json(self):
        return {"text": self.text, "tokens": [
            {"id": t.i, "start": t.idx, "end": t.idx + len(t.text), "text": t.text,
             "lemma": t.lemma_, "pos": t.pos_, "tag": t.tag_, "morph": str(t.morph)}
            for t in self._tokens]}

    def __repr__(self):
        return self.text

    def __str__(self):
        return self.text

## 2.4 Tokenizador

El tokenizador recorre el texto con **una única expresión regular** cuyas alternativas se
prueban en orden de prioridad (la primera que encaja gana):

1. **URLs y emails** → un único token.
2. **Excepciones / abreviaturas** (`ekz.`, `k.t.p.`, `s-ro`, `km/h`...) → un único token, para que
   el punto no se confunda con un final de frase.
3. **Elisión del artículo** `l'` (`de l' mondo`).
4. **Fechas** ISO (`2007-03-10`) y **números**: `1950`, `1.435`, `3,5`, y números con terminación gramatical
   (`la 20-a jarcento` = el siglo XX, `la 1950-aj jaroj` = los años 50).
5. **Palabras**: letras Unicode (incluye `ĉ ĝ ĥ ĵ ŝ ŭ` y el sistema-x `cx, gx...`), permitiendo
   compuestos con guion (`Dizel-lokomotivo`) y alfanuméricos (`A380`, `TGV`).
6. **Puntos suspensivos** `...` / `…`.
7. Cualquier otro carácter no-espacio → token de **puntuación o símbolo**.

Cada token guarda su `idx` (offset) y el espacio que le sigue (`whitespace_`), por lo que
`"".join(t.text_with_ws for t in doc)` reconstruye el texto original.

In [ ]:
class Tokenizer:
    """Tokenizador basado en expresiones regulares con excepciones (como spaCy)."""

    def __init__(self, exceptions=None):
        self.exceptions = sorted(set(exceptions or MALLONGIGOJ), key=len, reverse=True)
        self._compile()

    def add_special_case(self, text):
        """Añade una excepción (p. ej. una abreviatura) que nunca se dividirá."""
        self.exceptions = sorted(set(self.exceptions) | {text}, key=len, reverse=True)
        self._compile()

    def _compile(self):
        litero = r"[^\W\d_]"                 # cualquier letra Unicode
        alnum = r"[^\W_]"
        exc = "|".join(re.escape(e) for e in self.exceptions)
        self.pattern = re.compile(
            r"(?P<url>(?:https?://|www\.)\S+[^\s.,;:!?)\]»”\"']|[\w.+-]+@[\w-]+\.[\w.]+)"
            rf"|(?P<exc>(?<!{alnum})(?i:{exc})(?!{alnum}))"
            rf"|(?P<elizio>(?<!{alnum})[lL]['’](?=\s|$))"
            r"|(?P<dato>\d{4}-\d{2}-\d{2})"
            r"|(?P<num>\d+(?:[.,]\d+)*(?:-[a-zĉĝĥĵŝŭ]+)?)"
            rf"|(?P<vorto>{litero}{alnum}*(?:[-'’]{alnum}+)*)"
            r"|(?P<tripunkto>\.{3}|…)"
            r"|(?P<signo>\S)"
        )

    def __call__(self, text):
        tokens = []
        matches = list(self.pattern.finditer(text))
        for k, m in enumerate(matches):
            fin = matches[k + 1].start() if k + 1 < len(matches) else len(text)
            tokens.append((m.group(), m.start(), text[m.end():fin]))
        return Doc(text, tokens)

    def explain(self, text):
        """Muestra qué regla ha generado cada token (útil para depurar)."""
        return [(m.lastgroup, m.group()) for m in self.pattern.finditer(text)]

## 2.5 Componentes del pipeline

### 2.5.1 Normalizador
Rellena `token.norm_`:
* minúsculas y normalización Unicode **NFC** (para que `ŝ` precompuesta y `s + ̂` sean iguales);
* apóstrofo tipográfico `’` → `'`; `ǔ` → `ŭ`;
* **sistema-x**: muchos textos en Internet escriben `cx, gx, hx, jx, sx, ux` en lugar de
  `ĉ, ĝ, ĥ, ĵ, ŝ, ŭ` (porque no tenían el teclado). Como la `x` no existe en el alfabeto
  del Esperanto, la conversión es segura para palabras esperantistas. Se aplica solo a tokens
  que **no** empiezan en mayúscula a mitad de frase, para no estropear nombres extranjeros (`Linux`).

In [ ]:
X_SISTEMO = {"cx": "ĉ", "gx": "ĝ", "hx": "ĥ", "jx": "ĵ", "sx": "ŝ", "ux": "ŭ"}
_X_RE = re.compile(r"[cghjsu]x")


class Normalizer:
    name = "normalizer"

    def __init__(self, x_sistemo=True):
        self.x_sistemo = x_sistemo

    @staticmethod
    def normalizi(text, x_sistemo=True):
        t = unicodedata.normalize("NFC", text).lower().replace("’", "'").replace("ǔ", "ŭ")
        if x_sistemo and "x" in t:
            t = _X_RE.sub(lambda m: X_SISTEMO[m.group()], t)
        return t

    def __call__(self, doc):
        for i, t in enumerate(doc):
            propra = t.text[:1].isupper() and i > 0
            t.norm_ = self.normalizi(t.text, self.x_sistemo and not propra)
        return doc

### 2.5.2 Segmentador de frases (`sentencizer`)
Marca `token.is_sent_start`. Una frase empieza:
* en el primer token;
* tras `.`, `!`, `?`, `…` (o una abreviatura final como `ktp.`), saltando comillas/paréntesis
  de cierre, **si** el siguiente token empieza por mayúscula, dígito o comilla de apertura;
* tras un salto de línea (útil para el texto de páginas web).

In [ ]:
class Sentencizer:
    name = "sentencizer"
    FINOJ = {".", "!", "?", "…", "..."}
    FERMAJ = {")", "]", "»", "”", '"', "'", "’"}
    MALFERMAJ = {"(", "«", "„", "“", '"', "'"}

    def __call__(self, doc):
        toks = list(doc)
        for t in toks:
            t.is_sent_start = False
        if toks:
            toks[0].is_sent_start = True
        i = 0
        while i < len(toks) - 1:
            t = toks[i]
            if "\n" in t.whitespace_:
                toks[i + 1].is_sent_start = True
            elif t.text in self.FINOJ or t.norm_ in MALLONGIGOJ_FINAJ:
                j = i + 1
                while j < len(toks) and toks[j].text in self.FERMAJ and "\n" not in toks[j - 1].whitespace_:
                    j += 1
                if j < len(toks):
                    nxt = toks[j].text[:1]
                    if nxt.isupper() or nxt.isdigit() or nxt in self.MALFERMAJ:
                        toks[j].is_sent_start = True
                i = j - 1
            i += 1
        return doc

### 2.5.3 Analizador morfológico (núcleo del modelo)

La función `analizi(norm)` analiza **una palabra aislada** y devuelve `(pos, lema, rasgos, tag, raíz)`.
El orden de las reglas importa:

1. **Léxico cerrado** (pronombres, correlativos, preposiciones...).
2. **Números** en cifras (`1950`, `20-a`) y numerales compuestos (`dudek`, `tricent`).
3. **Palabras extranjeras**: si contienen letras que no existen en el alfabeto esperanto
   (`q, w, x, y, ü, é, ó`...) → `X` (o `PROPN` si van en mayúscula). Detecta, p. ej., el texto en
   español/inglés/alemán que se cuela en Wikipedia a través de las referencias.
4. **Participios** (`-anta`, `-inta`, `-ita`...). Riesgo: palabras como `ŝtato`, `aparato`,
   `konstanta` o `piloto` *parecen* participios pero no lo son. Para minimizar errores:
   * solo se reducen al verbo los **adjetivos y adverbios** (`konstruita → konstrui`), no los sustantivos
     (en `-ato/-ito/-oto` casi siempre son raíces: `ŝtato`, `aparato`, `piloto`);
   * la raíz restante debe tener ≥ 2 letras (`uzata → uzi`) y no estar en una lista de **excepciones**
     (también dentro de compuestos: `senpilota`, `post-milita`).
5. **Verbos** (`-as, -is, -os, -us, -u, -i`).
6. **Sustantivos** (`-o, -oj, -on, -ojn`), **adjetivos** (`-a, -aj, -an, -ajn`),
   **adverbios** (`-e, -en`).
7. Si nada encaja → `X` (palabra extranjera, sigla...).

Se usa `lru_cache`: cada forma distinta se analiza una sola vez, lo que acelera mucho el
procesamiento del corpus (el vocabulario se repite).

**Lematización**: gracias a la regularidad, lematizar es *sustituir la terminación flexiva por la
terminación de diccionario*: `-ojn → -o`, `-ajn → -a`, `-is → -i`, `-en → -e`, `-ita → -i`...

**Raíz (`token._.radiko`)**: el lema sin su vocal final (`lokomotivo → lokomotiv`). Agrupa familias
de palabras (`elektro`, `elektra`, `elektre` → `elektr`), lo que será útil en la Fase 3.

In [ ]:
# Raíces que terminan como un participio pero NO lo son (falsos positivos detectados en el corpus)
NE_PARTICIPOJ = {
    "konst", "gig", "eleg", "elef", "horiz", "aŭtom", "priv", "apar", "klim", "sold", "kandid",
    "rezult", "sen", "kval", "pil", "rob", "patri", "idi", "diam", "mil", "parazi", "str",
    "ril", "jezui", "fosf", "merk", "adiab", "bo", "profi", "sport", "kroko", "demokr",
    # raíces de 2 letras (ŝtata, fronta, brita, planta...)
    "ŝt", "fr", "br", "gr", "pl", "sp", "tr", "kv", "fl", "oh",
}
# Prefijos/preposiciones que pueden preceder a la raíz en un compuesto (sen-pilota, post-milita)
_KUNMETAĴAJ_PREFIKSOJ = ("sen", "sur", "post", "ekster", "mal", "ne", "antaŭ", "inter", "sub",
                         "super", "tra", "trans", "pri", "kontraŭ", "kun", "el", "en")


def _ne_participo(r):
    """True si la raíz r (o r sin un prefijo de compuesto) es un falso participio."""
    r = r.split("-")[-1]
    if r in NE_PARTICIPOJ:
        return True
    return any(r.startswith(p) and r[len(p):] in NE_PARTICIPOJ for p in _KUNMETAĴAJ_PREFIKSOJ)


PARTICIPO_RE = re.compile(r"^(?P<r>.{2,}?)(?P<s>ant|int|ont|at|it|ot)(?P<f>a|aj|an|ajn|e|en|o|oj|on|ojn)$")
VERBO_RE = re.compile(r"^(?P<r>.{2,}?)(?P<f>as|is|os|us|u|i)$")
O_RE = re.compile(r"^(?P<r>.+?)(?P<f>o|oj|on|ojn)$")
A_RE = re.compile(r"^(?P<r>.+?)(?P<f>a|aj|an|ajn)$")
E_RE = re.compile(r"^(?P<r>.+?)(?P<f>e|en)$")
# Alfabeto del Esperanto (28 letras). Una palabra con otras letras (q, w, x, y, ü, é, ó...) es extranjera.
ALFABETO = set("abcĉdefgĝhĥijĵklmnoprsŝtuŭvz")
CIFERO_RE = re.compile(r"^\d+(?:[.,]\d+)*$")
DATO_RE = re.compile(r"^\d{4}-\d{2}-\d{2}$")
CIFERO_FINAĴO_RE = re.compile(r"^\d+-(?P<f>a|aj|an|ajn|e|o|oj|on|ojn)$")

PART_TEMPO = {"ant": ("Pres", "Act"), "int": ("Past", "Act"), "ont": ("Fut", "Act"),
              "at": ("Pres", "Pass"), "it": ("Past", "Pass"), "ot": ("Fut", "Pass")}
VERBO_FINOJ = {"as": {"Tense": "Pres", "Mood": "Ind", "VerbForm": "Fin"},
               "is": {"Tense": "Past", "Mood": "Ind", "VerbForm": "Fin"},
               "os": {"Tense": "Fut", "Mood": "Ind", "VerbForm": "Fin"},
               "us": {"Mood": "Cnd", "VerbForm": "Fin"},
               "u":  {"Mood": "Imp", "VerbForm": "Fin"},     # volitivo/imperativo
               "i":  {"VerbForm": "Inf"}}


def _nombro_kazo(fino):
    """Rasgos de número y caso a partir de la terminación nominal/adjetival (-j, -n)."""
    return {"Number": "Plur" if "j" in fino else "Sing", "Case": "Acc" if fino.endswith("n") else "Nom"}


def _estas_nombro(w):
    return w in NOMBROJ_BAZAJ or bool(NOMBRO_RE.match(w))


@lru_cache(maxsize=200_000)
def analizi(w):
    """Analiza una palabra normalizada. Devuelve (pos, lema, rasgos(dict), tag, radiko)."""
    # 1) Léxico de palabras cerradas
    if w in LEKSIKONO:
        pos, lemo, feats, tag = LEKSIKONO[w]
        return pos, lemo, feats, tag, lemo

    # 2) Números
    if DATO_RE.match(w):
        return "NUM", w, {"NumType": "Card"}, "dato", w
    if CIFERO_RE.match(w):
        return "NUM", w, {"NumType": "Card"}, "cifero", w
    m = CIFERO_FINAĴO_RE.match(w)
    if m:                                      # 20-a, 1950-aj (ordinales escritos con cifras)
        f = m.group("f")
        base = w[: -len(f)] + f[0]
        pos = {"a": "ADJ", "e": "ADV", "o": "NOUN"}[f[0]]
        feats = {"NumType": "Ord"} if pos != "ADV" else {}
        if pos != "ADV":
            feats.update(_nombro_kazo(f))
        return pos, base, feats, "-" + f, base
    if _estas_nombro(w):
        return "NUM", w, {"NumType": "Card"}, "nombro", w

    # 3) Palabras extranjeras: contienen letras ajenas al alfabeto (aviación, München, Wayback)
    if any(c.isalpha() and c not in ALFABETO for c in w):
        return "X", w, {"Foreign": "Yes"}, "fremda", w

    # 4) Participios (solo adjetivales/adverbiales se reducen al verbo)
    m = PARTICIPO_RE.match(w)
    if m and not _ne_participo(m.group("r")):
        r, s, f = m.group("r"), m.group("s"), m.group("f")
        tense, voice = PART_TEMPO[s]
        if f[0] in "ae":
            feats = {"VerbForm": "Part" if f[0] == "a" else "Conv", "Tense": tense, "Voice": voice}
            if f[0] == "a":
                feats.update(_nombro_kazo(f))
                return "ADJ", r + "i", feats, f"-{s}-{f}", r
            return "ADV", r + "i", feats, f"-{s}-{f}", r
        if s in ("ant", "int", "ont"):          # veturanto (conductor), loĝanto (habitante)
            feats = {"VerbForm": "Part", "Tense": tense, "Voice": voice, **_nombro_kazo(f)}
            return "NOUN", r + s + "o", feats, f"-{s}-{f}", r

    # 5) Verbos
    m = VERBO_RE.match(w)
    if m:
        r, f = m.group("r"), m.group("f")
        pos = "AUX" if r + "i" in HELPVERBOJ else "VERB"
        return pos, r + "i", dict(VERBO_FINOJ[f]), "-" + f, r

    # 6) Sustantivos, adjetivos y adverbios derivados
    m = O_RE.match(w)
    if m:
        return "NOUN", m.group("r") + "o", _nombro_kazo(m.group("f")), "-" + m.group("f"), m.group("r")
    m = A_RE.match(w)
    if m:
        r = m.group("r")
        feats = _nombro_kazo(m.group("f"))
        if _estas_nombro(r) or r == "unu":     # unua, dua, dudeka -> ordinales
            feats["NumType"] = "Ord"
        return "ADJ", r + "a", feats, "-" + m.group("f"), r
    m = E_RE.match(w)
    if m:
        feats = {"Case": "Acc"} if m.group("f") == "en" else {}   # -en = dirección (hejmen)
        return "ADV", m.group("r") + "e", feats, "-" + m.group("f"), m.group("r")
    if w.endswith("aŭ") and len(w) > 3:        # adverbios primitivos no listados
        return "ADV", w, {}, "-aŭ", w

    # 7) Desconocida (palabra extranjera, sigla...)
    return "X", w, {"Foreign": "Yes"} if w.isalpha() else {}, "nekonata", w

### 2.5.4 Etiquetador (`tagger`) — reglas de contexto

Primero asigna a cada token el análisis aislado de `analizi()` y después aplica
**reglas contextuales** que una palabra suelta no puede resolver:

| Regla | Ejemplo |
|---|---|
| Puntuación y símbolos → `PUNCT` / `SYM` | `,` `.` / `%` `€` |
| Sustantivo, verbo o palabra extranjera en mayúscula **no** al inicio de frase → `PROPN` | `en Francio`, `de Airbus`, `la Boeing 747` |
| ...salvo adjetivo en mayúscula que concuerda con el sustantivo siguiente → `ADJ` | `la Jetmotora aviadilo` vs. `de Toyota` |
| Siglas (todo mayúsculas) → `PROPN` | `TGV`, `NASA` |
| Correlativo en `-u` seguido de sustantivo/adjetivo → `DET` (si no, `PRON`) | `tiu trajno` vs. `tiu, kiu venis` |
| `dum`/`ĝis` + verbo conjugado cercano → `SCONJ` (si no, `ADP`) | `dum la trajno veturis` vs. `dum la milito` |
| `povi/devi/voli` + infinitivo → `AUX` (si no, `VERB`) | `povas veturi` vs. `ili volas pacon` |
| `kiel` + sustantivo/pronombre → `ADP` ("como") | `kiel piloto` |

In [ ]:
SIMBOLOJ = set("%€$£¥+=<>°×÷&#@§©®™/\\|~^*")


class Tagger:
    name = "tagger"

    def __call__(self, doc):
        toks = list(doc)
        for t in toks:
            if not any(c.isalnum() for c in t.text):          # puntuación o símbolo
                t.pos_ = "SYM" if t.text in SIMBOLOJ else "PUNCT"
                t.tag_ = t.pos_.lower()
                t.morph = MorphAnalysis({"PunctType": "Brck"} if t.text in "()[]{}" else {})
                continue
            if t.like_url:
                t.pos_, t.tag_, t.morph = "X", "url", MorphAnalysis()
                continue
            pos, _lemo, feats, tag, _r = analizi(t.norm_)
            t.pos_, t.tag_, t.morph = pos, tag, MorphAnalysis(feats)
            # Nombres propios: mayúscula a mitad de frase o siglas
            if t.norm_ not in LEKSIKONO and t.text[:1].isupper():
                sigla = t.is_upper and len(t.text) >= 2
                if sigla or pos == "X" or (not t.is_sent_start and pos in ("NOUN", "ADJ", "VERB")):
                    t.pos_ = "PROPN"
                    if not sigla and pos in ("NOUN", "ADJ"):
                        t.morph = MorphAnalysis({k: v for k, v in feats.items() if k in ("Number", "Case")})
                    else:
                        t.morph = MorphAnalysis({})
        self._kunteksto(toks)
        return doc

    @staticmethod
    def _kongruas(a, b):
        return a.morph.get("Number") == b.morph.get("Number") and a.morph.get("Case") == b.morph.get("Case")

    @staticmethod
    def _kunteksto(toks):
        n = len(toks)
        for i, t in enumerate(toks):
            nxt = toks[i + 1] if i + 1 < n else None
            # Adjetivo en mayúscula seguido de un sustantivo con el que concuerda: sigue siendo
            # adjetivo ("la Jetmotora aviadilo"); si no, es un nombre propio ("Toyota produktas").
            if t.pos_ == "PROPN" and t.tag_.startswith("-a") and not t.is_upper:
                if nxt is not None and nxt.pos_ in ("NOUN", "PROPN") and t.tag_[1:].replace("a", "o", 1) == nxt.tag_[1:]:
                    t.pos_ = "ADJ"
                    t.morph = MorphAnalysis(analizi(t.norm_)[2])
            # Correlativos en -u: determinante si acompañan a un nombre/adjetivo
            if t.tag_ == "korelativo" and t.norm_.rstrip("jn").endswith("u"):
                t.pos_ = "DET" if nxt is not None and nxt.pos_ in ("NOUN", "ADJ", "PROPN") else "PRON"
            # dum / ĝis: conjunción si introduce una oración con verbo conjugado
            # Ámbito = tokens hasta la siguiente puntuación. Si contiene un verbo conjugado es una
            # oración subordinada... salvo que 'dum' abra la frase y el ámbito no se cierre con
            # coma: entonces ese verbo es el de la oración principal ("Dum la milito oni detruis...").
            elif t.norm_ in ("dum", "ĝis"):
                j = i + 1
                while j < n and toks[j].pos_ != "PUNCT" and not toks[j].is_sent_start:
                    j += 1
                verbo = any(u.pos_ in ("VERB", "AUX") and "Fin" in u.morph.get("VerbForm")
                            for u in toks[i + 1: j])
                fermita_per_komo = j < n and toks[j].text == ","
                if verbo and (not t.is_sent_start or fermita_per_komo):
                    t.pos_ = "SCONJ"
            # Verbos modales: AUX solo si van seguidos de infinitivo
            elif t.pos_ == "AUX" and t.norm_[:-2] + "i" != "esti" and t.norm_[:-1] + "i" != "esti":
                sekvaj = [u for u in toks[i + 1: i + 4] if u.pos_ not in ("ADV", "PART")]
                if not (sekvaj and "Inf" in sekvaj[0].morph.get("VerbForm")):
                    t.pos_ = "VERB"
            # kiel + sustantivo/pronombre = "como" (preposición)
            elif t.norm_ == "kiel" and nxt is not None and nxt.pos_ in ("NOUN", "PROPN", "PRON"):
                t.pos_ = "ADP"

### 2.5.5 Lematizador y gestor de stop-words
* El **lematizador** usa el mismo análisis cacheado y rellena `lemma_` y la extensión `_.radiko`.
  Para los nombres propios conserva la mayúscula y solo elimina el acusativo (`Francion → Francio`).
* El componente de **stop-words** marca `is_stop` consultando `nlp.Defaults.stop_words`, tanto
  por la forma como por el lema. Al ser un conjunto compartido, se puede modificar en caliente
  (`nlp.Defaults.stop_words.add(...)`), igual que en spaCy.

In [ ]:
Token.set_extension("radiko", default="", force=True)


class Lemmatizer:
    name = "lemmatizer"

    def __call__(self, doc):
        for t in doc:
            if t.pos_ in ("PUNCT", "SYM") or t.like_url:
                t.lemma_ = t.text
                t._.radiko = t.text
                continue
            _pos, lemo, _f, _tag, radiko = analizi(t.norm_)
            if t.pos_ == "PROPN":
                lemo = t.text
                if re.search(r"(o|oj)n$", lemo):
                    lemo = lemo[:-1]
                if lemo.endswith("oj"):
                    lemo = lemo[:-1]
                radiko = lemo.lower()
            t.lemma_ = lemo
            t._.radiko = radiko
        return doc


class StopWords:
    name = "stopwords"

    def __init__(self, stop_words):
        self.stop_words = stop_words

    def __call__(self, doc):
        for t in doc:
            t.is_stop = t.norm_ in self.stop_words or t.lemma_.lower() in self.stop_words
        return doc

## 2.6 La clase `Esperanto` (equivalente a `spacy.Language`)

Orquesta el pipeline. Permite inspeccionar y modificar los componentes (`pipe_names`,
`add_pipe`, `remove_pipe`, `select_pipes`) y procesar muchos textos con `nlp.pipe()`.

In [ ]:
class Esperanto:
    """Modelo de NLP para Esperanto basado en reglas, con API similar a spacy.Language."""

    lang = "eo"

    class Defaults:
        stop_words = HALTVORTOJ           # compartido, modificable (como spaCy)

    def __init__(self, x_sistemo=True):
        self.tokenizer = Tokenizer()
        self._pipeline = []
        self._disabled = set()
        self.add_pipe(Normalizer(x_sistemo))
        self.add_pipe(Sentencizer())
        self.add_pipe(Tagger())
        self.add_pipe(Lemmatizer())
        self.add_pipe(StopWords(self.Defaults.stop_words))
        self.meta = {"lang": "eo", "name": "esperantilo_nlp", "version": "1.0.0",
                     "description": "Modelo de NLP basado en reglas para Esperanto"}

    # --- Gestión del pipeline ---------------------------------------------
    @property
    def pipeline(self):
        return [(n, c) for n, c in self._pipeline if n not in self._disabled]

    @property
    def pipe_names(self):
        return [n for n, _ in self.pipeline]

    @property
    def component_names(self):
        return [n for n, _ in self._pipeline]

    def add_pipe(self, component, name=None, before=None, after=None, first=False):
        """Añade un componente: cualquier callable doc -> doc."""
        name = name or getattr(component, "name", None) or component.__name__
        if name in self.component_names:
            raise ValueError(f"Ya existe un componente llamado '{name}'")
        pos = len(self._pipeline)
        if first:
            pos = 0
        elif before:
            pos = self.component_names.index(before)
        elif after:
            pos = self.component_names.index(after) + 1
        self._pipeline.insert(pos, (name, component))
        return component

    def remove_pipe(self, name):
        i = self.component_names.index(name)
        return self._pipeline.pop(i)

    def get_pipe(self, name):
        return dict(self._pipeline)[name]

    def has_pipe(self, name):
        return name in self.component_names

    @contextmanager
    def select_pipes(self, disable=()):
        """with nlp.select_pipes(disable=['tagger']): ... (desactiva temporalmente)."""
        disable = {disable} if isinstance(disable, str) else set(disable)
        anterior = set(self._disabled)
        self._disabled |= disable
        try:
            yield self
        finally:
            self._disabled = anterior

    # --- Procesamiento -------------------------------------------------------
    def make_doc(self, text):
        return self.tokenizer(text)

    def __call__(self, text):
        doc = self.make_doc(text)
        for _name, proc in self.pipeline:
            doc = proc(doc)
        return doc

    def pipe(self, texts, batch_size=1000):
        """Procesa un iterable de textos de forma perezosa (generador)."""
        for text in texts:
            yield self(text)

    def __repr__(self):
        return f"<Esperanto lang='eo' pipeline={self.pipe_names}>"


def load(name="eo", **kwargs):
    """Equivalente a spacy.load(): devuelve el modelo listo para usar."""
    return Esperanto(**kwargs)


GLOSARIO = {
    "ADJ": "adjetivo (-a)", "ADP": "preposición", "ADV": "adverbio (-e o primitivo)",
    "AUX": "verbo auxiliar (esti, povi, devi, voli)", "CCONJ": "conjunción coordinante",
    "DET": "determinante (la, tiu, mia...)", "INTJ": "interjección", "NOUN": "sustantivo (-o)",
    "NUM": "numeral", "PART": "partícula", "PRON": "pronombre", "PROPN": "nombre propio",
    "PUNCT": "puntuación", "SCONJ": "conjunción subordinante", "SYM": "símbolo",
    "VERB": "verbo", "X": "otro / palabra extranjera",
    "korelativo": "palabra de la tabla de correlativos", "nekonata": "terminación no reconocida",
    "Acc": "acusativo (-n)", "Nom": "nominativo", "Plur": "plural (-j)", "Sing": "singular",
    "Part": "participio", "Conv": "participio adverbial (-ante, -inte...)", "Inf": "infinitivo (-i)",
    "Cnd": "condicional (-us)", "Imp": "volitivo/imperativo (-u)",
}


def explain(term):
    """Equivalente a spacy.explain()."""
    return GLOSARIO.get(term)


nlp = load("eo")
nlp

## 2.7 Demostración del API
### Análisis token a token

In [ ]:
doc = nlp("La rapidaj elektraj trajnoj veturis inter Parizo kaj Lyon, kaj la "
          "konstruita flughaveno povos akcepti multajn aviadilojn.")
doc.to_dataframe()

### Atributos al estilo spaCy

In [ ]:
for token in doc[:6]:
    print(f"{token.text:<12} lemma={token.lemma_:<10} pos={token.pos_:<6} tag={token.tag_:<6} "
          f"morph={str(token.morph):<30} stop={token.is_stop!s:<5} shape={token.shape_}")

print("\nexplain('Acc') ->", explain("Acc"))
print("Morph de 'trajnoj':", doc[3].morph, "| Number =", doc[3].morph.get("Number"))
print("Span doc[1:4]      :", doc[1:4], "| raíz:", doc[1:4].root)
print("Noun chunks        :", list(doc.noun_chunks))
print("Pipeline           :", nlp.pipe_names)

### Frases, correlativos, pronombres, sistema-x y abreviaturas

In [ ]:
ekzemploj = [
    "Tiu aŭto, kiun mi aĉetis, estas rapida. Ĉu vi vidis ĝin? Ili veturas hejmen.",
    "La trajno de Madrido al Barcelono veturas je 300 km/h ekz. en la 21-a jarcento.",
    "Cxiuj sxoforoj devas konduki auxtomobilojn singarde.",   # sistema-x
    "Dum la trajno veturis, la pasaĝeroj dormis. Dum la milito oni detruis la fervojon.",
]
for teksto in ekzemploj:
    d = nlp(teksto)
    print(teksto)
    for s in d.sents:
        print("   frase:", " ".join(f"{t.text}/{t.pos_}/{t.lemma_}" for t in s))
    print()

### Extensiones y modificación del pipeline
Igual que en spaCy, se pueden añadir atributos propios y componentes propios.
Como ejemplo, un componente que marca las palabras con el sufijo `-il-` (*herramienta*:
`aviadilo` = avión, literalmente "herramienta para volar").

In [ ]:
Token.set_extension("estas_ilo", default=False, force=True)


def ilo_detektilo(doc):
    for t in doc:
        t._.estas_ilo = t.pos_ == "NOUN" and t._.radiko.endswith("il")
    return doc


nlp.add_pipe(ilo_detektilo, name="ilo_detektilo", after="lemmatizer")
print(nlp.pipe_names)
d = nlp("La aviadilo kaj la veturilo uzas multajn ilojn.")
print([(t.text, t._.radiko, t._.estas_ilo) for t in d if t.pos_ == "NOUN"])
nlp.remove_pipe("ilo_detektilo")

with nlp.select_pipes(disable=["stopwords"]):
    print("Sin stop-words:", [t.is_stop for t in nlp("la trajno")], nlp.pipe_names)
print("Con stop-words:", [t.is_stop for t in nlp("la trajno")], nlp.pipe_names)

## 2.8 Evaluación del modelo NLP

### 2.8.1 Tests unitarios de fenómenos gramaticales
Cada `assert` comprueba una regla concreta; si alguna falla, el cuaderno se detiene.

In [ ]:
def analizo(teksto):
    return [(t.text, t.lemma_, t.pos_) for t in nlp(teksto)]


def kontroli(teksto, indekso, lemo=None, pos=None, morph=None):
    t = nlp(teksto)[indekso]
    if lemo is not None:
        assert t.lemma_ == lemo, f"{teksto!r}[{indekso}]: lema {t.lemma_!r} != {lemo!r}"
    if pos is not None:
        assert t.pos_ == pos, f"{teksto!r}[{indekso}]: pos {t.pos_!r} != {pos!r}"
    if morph is not None:
        for feat in morph.split("|"):
            assert feat in t.morph, f"{teksto!r}[{indekso}]: falta {feat} en {t.morph}"


testoj = [
    # Sustantivos y adjetivos: plural y acusativo
    ("Mi vidas rapidajn trajnojn.", 2, "rapida", "ADJ", "Case=Acc|Number=Plur"),
    ("Mi vidas rapidajn trajnojn.", 3, "trajno", "NOUN", "Case=Acc|Number=Plur"),
    # Tiempos verbales
    ("La aviadilo flugas.", 2, "flugi", "VERB", "Tense=Pres"),
    ("La aviadilo flugis.", 2, "flugi", "VERB", "Tense=Past"),
    ("La aviadilo flugos.", 2, "flugi", "VERB", "Tense=Fut"),
    ("La aviadilo flugus.", 2, "flugi", "VERB", "Mood=Cnd"),
    ("Flugu!", 0, "flugi", "VERB", "Mood=Imp"),
    # Participios
    ("La konstruita stacio", 1, "konstrui", "ADJ", "VerbForm=Part|Tense=Past|Voice=Pass"),
    ("La veturanta aŭto", 1, "veturi", "ADJ", "VerbForm=Part|Tense=Pres|Voice=Act"),
    ("La veturantoj", 1, "veturanto", "NOUN", "VerbForm=Part|Number=Plur"),
    ("Ŝi parolis ridante.", 2, "ridi", "ADV", "VerbForm=Conv"),
    # Falsos participios (no deben reducirse a verbo)
    ("La ŝtato kaj la aparato", 1, "ŝtato", "NOUN", None),
    ("La konstanta rapido", 1, "konstanta", "ADJ", None),
    ("La piloto flugas.", 1, "piloto", "NOUN", None),
    # Auxiliares
    ("Ĝi estas rapida.", 1, "esti", "AUX", None),
    ("Ni povas veturi.", 1, "povi", "AUX", None),
    ("Ni volas pacon.", 1, "voli", "VERB", None),
    # Pronombres y posesivos
    ("Mi vidis lin.", 2, "li", "PRON", "Case=Acc"),
    ("Iliajn aŭtojn", 0, "ilia", "DET", "Poss=Yes|Number=Plur|Case=Acc"),
    # Correlativos
    ("Tiujn aŭtojn", 0, "tiu", "DET", "Case=Acc|Number=Plur"),
    ("Kiu venis?", 0, "kiu", "PRON", None),
    ("Ili iris tien.", 2, "tie", "ADV", "Case=Acc"),
    # Adverbios derivados y dirección
    ("Ĝi veturas rapide.", 2, "rapide", "ADV", None),
    ("Ili iris hejmen.", 2, "hejme", "ADV", "Case=Acc"),
    # Números
    ("En 1950 estis du trajnoj.", 1, "1950", "NUM", None),
    ("La dudek aviadiloj", 1, "dudek", "NUM", None),
    ("La dua stacio", 1, "dua", "ADJ", "NumType=Ord"),
    ("En la 20-a jarcento", 2, "20-a", "ADJ", "NumType=Ord"),
    # Nombres propios
    ("Ĝi flugis al Francion.", 3, "Francio", "PROPN", None),
    ("La TGV estas rapida.", 1, "TGV", "PROPN", None),
    ("La Jetmotora aviadilo", 1, "jetmotora", "ADJ", None),
    ("Ĝin produktas Toyota kaj Honda.", 2, "Toyota", "PROPN", None),
    ("Ĝi estas la plej uzata aviadilo.", 4, "uzi", "ADJ", "Voice=Pass"),
    ("La brita fervojo", 1, "brita", "ADJ", None),
    ("La trajno Transrapid München", 3, "München", "PROPN", None),
    ("aviones y trenes", 1, "y", "X", "Foreign=Yes"),
    # Sistema-x
    ("Cxiuj sxoforoj", 1, "ŝoforo", "NOUN", None),
    # Preposición vs conjunción
    ("Dum la milito ili veturis.", 0, "dum", "ADP", None),
    ("Dum la trajno veturis, ni dormis.", 0, "dum", "SCONJ", None),
]
for teksto, i, lemo, pos, morph in testoj:
    kontroli(teksto, i, lemo, pos, morph)

# Tokenización
assert [t.text for t in nlp("Ĝi veturis je 300 km/h, ekz. en Japanio.")] == \
    ["Ĝi", "veturis", "je", "300", "km/h", ",", "ekz.", "en", "Japanio", "."]
assert [t.text for t in nlp("La Dizel-lokomotivo de l' fervojo")] == \
    ["La", "Dizel-lokomotivo", "de", "l'", "fervojo"]
assert [t.text for t in nlp("Vidu https://eo.wikipedia.org/wiki/Fervojo.")] == \
    ["Vidu", "https://eo.wikipedia.org/wiki/Fervojo", "."]
assert [t.text for t in nlp("Arkivigite je 2007-03-10.")] == ["Arkivigite", "je", "2007-03-10", "."]
# Reconstrucción exacta del texto original
_t = "La trajno,  kiu  veturis (rapide)... alvenis!"
assert "".join(t.text_with_ws for t in nlp(_t)) == _t
# Segmentación en frases (la abreviatura 'ekz.' no corta la frase)
assert len(list(nlp("Mi vidis ekz. trajnon. Poste mi vidis aviadilon! Ĉu vi?").sents)) == 3
# Stop-words por lema
assert [t.is_stop for t in nlp("Estis tiuj trajnoj")] == [True, True, False]

print(f"✔ {len(testoj) + 7} tests superados")

### 2.8.2 Exactitud sobre corpus anotados a mano (*gold standard*)

Formato de anotación: `palabra/POS/lema`. Se usan **dos** conjuntos:

1. **Frases propias** (10 frases, ~110 tokens) escritas por el equipo con fenómenos variados.
   Se usaron *durante el desarrollo*, así que su resultado es **optimista**.
2. **Frases reales del dataset** (7 frases, ~150 tokens) escogidas al azar (`random.seed(99)`)
   **después** de cerrar las reglas y anotadas a mano **sin volver a tocar el modelo**.
   Es la estimación honesta del rendimiento en texto real de Wikipedia.

In [ ]:
ORO_PROPRA = """
La/DET/la unua/ADJ/unua fervojo/NOUN/fervojo por/ADP/por pasaĝeroj/NOUN/pasaĝero estis/AUX/esti malfermita/ADJ/malfermi en/ADP/en Anglio/PROPN/Anglio en/ADP/en 1825/NUM/1825 ./PUNCT/.
Modernaj/ADJ/moderna rapidtrajnoj/NOUN/rapidtrajno povas/AUX/povi veturi/VERB/veturi pli/ADV/pli rapide/ADV/rapide ol/SCONJ/ol aŭtomobiloj/NOUN/aŭtomobilo ./PUNCT/.
La/DET/la piloto/NOUN/piloto kondukis/VERB/konduki la/DET/la aviadilon/NOUN/aviadilo al/ADP/al la/DET/la flughaveno/NOUN/flughaveno de/ADP/de Madrido/PROPN/Madrido ./PUNCT/.
Multaj/ADJ/multa homoj/NOUN/homo preferas/VERB/preferi vojaĝi/VERB/vojaĝi per/ADP/per aŭtobuso/NOUN/aŭtobuso ĉar/SCONJ/ĉar ĝi/PRON/ĝi estas/AUX/esti malmultekosta/ADJ/malmultekosta ./PUNCT/.
Kiam/ADV/kiam la/DET/la motoro/NOUN/motoro ekfunkciis/VERB/ekfunkcii ,/PUNCT/, la/DET/la ŝoforo/NOUN/ŝoforo rimarkis/VERB/rimarki ke/SCONJ/ke la/DET/la bremsoj/NOUN/bremso ne/PART/ne funkcias/VERB/funkcii ./PUNCT/.
Elektraj/ADJ/elektra lokomotivoj/NOUN/lokomotivo ricevas/VERB/ricevi energion/NOUN/energio el/ADP/el supra/ADJ/supra kontaktdrato/NOUN/kontaktdrato ./PUNCT/.
Tiuj/DET/tiu helikopteroj/NOUN/helikoptero flugos/VERB/flugi super/ADP/super la/DET/la montoj/NOUN/monto dum/ADP/dum tri/NUM/tri horoj/NOUN/horo ./PUNCT/.
La/DET/la flugkompanio/NOUN/flugkompanio aĉetis/VERB/aĉeti dudek/NUM/dudek novajn/ADJ/nova aviadilojn/NOUN/aviadilo de/ADP/de Airbus/PROPN/Airbus ./PUNCT/.
Ŝi/PRON/ŝi parkumis/VERB/parkumi sian/DET/sia aŭton/NOUN/aŭto apud/ADP/apud la/DET/la stacidomo/NOUN/stacidomo kaj/CCONJ/kaj iris/VERB/iri hejmen/ADV/hejme ./PUNCT/.
La/DET/la vagonoj/NOUN/vagono estas/AUX/esti tirataj/ADJ/tiri de/ADP/de dizela/ADJ/dizela lokomotivo/NOUN/lokomotivo ,/PUNCT/, kiu/PRON/kiu atingas/VERB/atingi 120/NUM/120 kilometrojn/NOUN/kilometro hore/ADV/hore ./PUNCT/.
"""

ORO_REALA = """
Ĝi/PRON/ĝi povas/AUX/povi esti/AUX/esti influita/ADJ/influi de/ADP/de potencotrajna/ADJ/potencotrajna dezajno/NOUN/dezajno ,/PUNCT/, lubrikaĵa/ADJ/lubrikaĵa speco/NOUN/speco kaj/CCONJ/kaj temperaturo/NOUN/temperaturo kaj/CCONJ/kaj multaj/ADJ/multa aliaj/ADJ/alia faktoroj/NOUN/faktoro ./PUNCT/.
Aviadiloj/NOUN/aviadilo povas/AUX/povi anstataŭe/ADV/anstataŭe uzi/VERB/uzi jetmotorojn/NOUN/jetmotoro kaj/CCONJ/kaj helikopteroj/NOUN/helikoptero povas/AUX/povi anstataŭe/ADV/anstataŭe uzi/VERB/uzi turboŝaftojn/NOUN/turboŝafto ,/PUNCT/, kiuj/PRON/kiu ambaŭ/PRON/ambaŭ estas/AUX/esti specoj/NOUN/speco de/ADP/de turbinoj/NOUN/turbino ./PUNCT/.
Oni/PRON/oni povas/AUX/povi la/DET/la sekurecon/NOUN/sekureco de/ADP/de la/DET/la aŭtomobiloj/NOUN/aŭtomobilo plibonigi/VERB/plibonigi per/ADP/per administraj/ADJ/administra kaj/CCONJ/kaj konstruaj/ADJ/konstrua rimedoj/NOUN/rimedo ./PUNCT/.
Harmon/PROPN/Harmon esprimis/VERB/esprimi favoron/NOUN/favoro al/ADP/al Esperanto/PROPN/Esperanto en/ADP/en la/DET/la franca/ADJ/franca Esperanto-revuo/NOUN/esperanto-revuo La/DET/la Movado/PROPN/Movado ./PUNCT/.
Selden/PROPN/Selden postulis/VERB/postuli patenton/NOUN/patento por/ADP/por aŭto/NOUN/aŭto en/ADP/en 1879/NUM/1879 ,/PUNCT/, sed/CCONJ/sed la/DET/la patentopostulo/NOUN/patentopostulo elvalidiĝis/VERB/elvalidiĝi ĉar/SCONJ/ĉar la/DET/la vehiklo/NOUN/vehiklo neniam/ADV/neniam estis/AUX/esti konstruita/ADJ/konstrui ./PUNCT/.
Tion/PRON/tio oni/PRON/oni provas/VERB/provi obstakli/VERB/obstakli per/ADP/per rentgenaj/ADJ/rentgena aparatoj/NOUN/aparato kaj/CCONJ/kaj hundoj/NOUN/hundo ĉe/ADP/ĉe la/DET/la limo/NOUN/limo aŭ/CCONJ/aŭ eĉ/ADV/eĉ landinterne/ADV/landinterne ./PUNCT/.
La/DET/la kapa/ADJ/kapa salono/NOUN/salono havas/VERB/havi 110/NUM/110 sidajn/ADJ/sida kaj/CCONJ/kaj 140/NUM/140 starajn/ADJ/stara lokojn/NOUN/loko ,/PUNCT/, la/DET/la dua/ADJ/dua salono/NOUN/salono –/PUNCT/– 60/NUM/60 sidajn/ADJ/sida kaj/CCONJ/kaj 40/NUM/40 starajn/ADJ/stara ./PUNCT/.
"""


def legi_oron(teksto):
    """Convierte el texto anotado en una lista de frases [(palabra, POS, lema), ...]."""
    return [[e.rsplit("/", 2) for e in linio.split(" ") if e] for linio in teksto.strip().splitlines()]


def taksi(frazoj):
    """Devuelve (n_tokens, accuracy POS, accuracy lema, lista de errores)."""
    tot = pos_ok = lem_ok = 0
    eraroj = []
    for eroj in frazoj:
        vortoj = [e[0] for e in eroj]
        doc = nlp(" ".join(vortoj))
        assert [t.text for t in doc] == vortoj, "La tokenización no coincide con la anotación"
        for t, (v, pos, lemo) in zip(doc, eroj):
            tot += 1
            pos_ok += t.pos_ == pos
            lem_ok += t.lemma_ == lemo
            if t.pos_ != pos or t.lemma_ != lemo:
                eraroj.append({"palabra": v, "predicho": f"{t.pos_} / {t.lemma_}",
                               "correcto": f"{pos} / {lemo}"})
    return tot, pos_ok / tot, lem_ok / tot, eraroj


rezultoj, ĉiuj_eraroj = [], []
for nomo, oro in (("Frases propias (desarrollo)", ORO_PROPRA),
                  ("Frases reales del dataset (no vistas)", ORO_REALA)):
    n_tok, acc_pos, acc_lem, eraroj = taksi(legi_oron(oro))
    rezultoj.append({"conjunto": nomo, "tokens": n_tok, "accuracy POS": f"{acc_pos:.1%}",
                     "accuracy lema": f"{acc_lem:.1%}"})
    ĉiuj_eraroj += [{"conjunto": nomo, **e} for e in eraroj]
display(pd.DataFrame(rezultoj))
pd.DataFrame(ĉiuj_eraroj)

**Análisis de errores.** Los fallos en texto real se concentran en casos que las reglas no
pueden resolver sin conocimiento del mundo o sin análisis sintáctico:
* **Nombres propios al inicio de frase** (`Harmon`, `Selden`): al ir en mayúscula por ser la
  primera palabra, no se distinguen de un sustantivo común; además `Selden` acaba en `-en` y se
  analiza como adverbio. Una solución sería un diccionario de nombres propios (*gazetteer*).
* **Mayúsculas «de título»** de Wikipedia (`Esperanto-revuo`) que se toman por nombre propio.
* **Ambigüedades sintácticas**: `povas` separado de su infinitivo por varias palabras
  (`Oni povas la sekurecon ... plibonigi`) o `ambaŭ` usado como pronombre.

Estos errores apenas afectan a la clasificación temática, que se basará en los lemas de
sustantivos, adjetivos y verbos.

## 2.9 Aplicación del modelo al dataset de la Fase 1

Procesamos todo el corpus con `nlp.pipe()` y medimos:
* **velocidad** de procesamiento;
* **distribución de categorías gramaticales** (y cuántos tokens quedan como `X` = no reconocidos);
* **proporción de stop-words**;
* **lemas más frecuentes por clase** (sin stop-words), como comprobación de que los lemas
  capturan el vocabulario temático.

In [ ]:
t0 = time.perf_counter()
dokumentoj = list(nlp.pipe(df["text"]))
daŭro = time.perf_counter() - t0
n_tokens = sum(len(d) for d in dokumentoj)
print(f"{len(dokumentoj)} párrafos · {n_tokens} tokens · {daŭro:.2f} s "
      f"({n_tokens / daŭro:,.0f} tokens/s)")

pos_kalk = Counter(t.pos_ for d in dokumentoj for t in d)
vortoj_tot = sum(v for k, v in pos_kalk.items() if k not in ("PUNCT", "SYM"))
print(f"Tokens no reconocidos (X): {pos_kalk['X']} ({pos_kalk['X'] / vortoj_tot:.1%} de las palabras)")
print(f"Stop-words: {sum(t.is_stop for d in dokumentoj for t in d) / vortoj_tot:.1%} de las palabras")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
etikedoj, valoroj = zip(*pos_kalk.most_common())
ax[0].bar(etikedoj, valoroj, color="#4C72B0")
ax[0].set_title("Categorías gramaticales (UPOS) en el corpus")
ax[0].tick_params(axis="x", rotation=45)
ax[0].set_ylabel("nº de tokens")

# Proporción de algunas categorías dentro de cada clase
pos_klaso = pd.DataFrame([{"class": k, "pos": t.pos_} for d, k in zip(dokumentoj, df["class"]) for t in d])
tab = pd.crosstab(pos_klaso["class"], pos_klaso["pos"], normalize="index")
tab[["NOUN", "ADJ", "VERB", "PROPN", "NUM", "X"]].plot.bar(ax=ax[1], rot=0)
ax[1].set_title("Proporción de algunas categorías por clase")
ax[1].set_ylabel("proporción de tokens")
plt.tight_layout()
plt.show()

### Palabras extranjeras: una medida del ruido del dataset
Las palabras con letras ajenas al alfabeto esperanto (`X` con `Foreign=Yes`) revelan párrafos
con texto en otros idiomas (normalmente títulos de referencias en español, inglés o alemán).
Es una forma de **medir parte del ruido** del dataset de la Fase 1.

In [ ]:
def fremda_proporcio(doc):
    """Proporción de palabras del documento analizadas como extranjeras/desconocidas (X)."""
    vortoj = [t for t in doc if t.is_alpha]
    return sum(t.pos_ == "X" for t in vortoj) / max(len(vortoj), 1)


df["prop_extranjeras"] = [fremda_proporcio(d) for d in dokumentoj]
bruaj = df[df["prop_extranjeras"] > 0.2]
print(f"Párrafos con >20% de palabras extranjeras: {len(bruaj)} de {len(df)}")
bruaj[["text", "class", "prop_extranjeras"]].head(8)

### Lemas más frecuentes por clase
Si el lematizador funciona, las formas `trajno, trajnoj, trajnon, trajnojn` cuentan como un
único lema `trajno`, y los lemas más frecuentes de cada clase deben ser claramente temáticos.

In [ ]:
SATURAJ_POS = {"NOUN", "ADJ", "VERB", "PROPN", "ADV"}   # categorías con contenido léxico

klasoj = sorted(df["class"].unique())
fig, axes = plt.subplots(1, len(klasoj), figsize=(16, 4.5))
for ax, klaso in zip(axes, klasoj):
    kalk = Counter(t.lemma_ for i in df.index[df["class"] == klaso] for t in dokumentoj[i]
                   if t.pos_ in SATURAJ_POS and not t.is_stop and len(t.lemma_) > 2)
    lemoj, n = zip(*kalk.most_common(15))
    ax.barh(lemoj[::-1], n[::-1], color="#55A868")
    ax.set_title(f"Clase «{klaso}»")
plt.suptitle("15 lemas más frecuentes por clase (sin stop-words)")
plt.tight_layout()
plt.show()

### Reducción del vocabulario gracias a la lematización
Comparamos el número de tipos distintos (vocabulario) usando la forma original, el lema y la raíz.
Un vocabulario más pequeño significa una matriz menos dispersa en la Fase 3.

In [ ]:
vortaro = {
    "forma (minúsculas)": {t.norm_ for d in dokumentoj for t in d if t.is_alpha},
    "lema": {t.lemma_.lower() for d in dokumentoj for t in d if t.is_alpha},
    "raíz (_.radiko)": {t._.radiko.lower() for d in dokumentoj for t in d if t.is_alpha},
}
for nomo, aro in vortaro.items():
    print(f"{nomo:<20}: {len(aro):>6} tipos distintos")

## 2.10 Preparación para la Fase 3: analizador para vectorizar

La función `analizilo(texto)` convierte un texto en la lista de **lemas significativos**:
se descartan stop-words, puntuación, símbolos, números (también ordinales como `20-a`), URLs y
palabras extranjeras.
Se puede pasar directamente a `CountVectorizer` / `TfidfVectorizer` de scikit-learn con
`analyzer=analizilo`, de modo que **toda** la tokenización y normalización la hace nuestro modelo.

In [ ]:
FORIGI_POS = {"PUNCT", "SYM", "NUM", "X"}


def analizilo(teksto, uzi_radikon=False):
    """Texto -> lista de lemas (o raíces) de contenido, en minúsculas."""
    doc = nlp(teksto)
    return [(t._.radiko if uzi_radikon else t.lemma_).lower() for t in doc
            if not t.is_stop and t.pos_ not in FORIGI_POS and not t.like_url
            and len(t.lemma_) > 1 and not t.lemma_[:1].isdigit()]   # fuera '20-a', '1950-aj'...


print(analizilo(df["text"].iloc[0]))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vektorigilo = TfidfVectorizer(analyzer=analizilo, min_df=2)
X_tfidf = vektorigilo.fit_transform(df["text"])
print("Matriz TF-IDF:", X_tfidf.shape, "(párrafos x lemas)")
print("Ejemplos de columnas:", list(vektorigilo.get_feature_names_out()[:15]))